# Testing for PropLenght

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import r2_score

import os
from helper.pipeline_utils import Net



def infer_prop_length(
    model_path: str,
    x_scaler_path: str,
    y_scaler_path: str,
    test_path: str,
    in_dim: int,
    out_dim: int,
    hidden_dims: list,
    activation: str,
):
    """
    Load scalers & model, run inference on test data, compute R²,
    and plot results.

    Args:
        model_path (str): Path to the saved model .pth file.
        x_scaler_path (str): Path to x_scaler.pkl.
        y_scaler_path (str): Path to y_scaler.pkl.
        test_path (str): Path to TestData.csv.
        in_dim (int): Number of input features.
        out_dim (int): Number of output targets.
        hidden_dims (list[int]): Hidden layer sizes.
        activation (str): Activation name ("relu", "sigmoid", or "softplus").
    """
    # [1] Load saved scalers
    with open(x_scaler_path, "rb") as f:
        x_scaler = pickle.load(f)
    with open(y_scaler_path, "rb") as f:
        y_scaler = pickle.load(f)

    # [2] Load and preprocess test data
    df_test = pd.read_csv(test_path)
    output_cols = df_test.columns[in_dim:].tolist()

    X_test = x_scaler.transform(df_test.iloc[:, :in_dim].values)
    Y_test_scaled = y_scaler.transform(df_test.iloc[:, in_dim:].values)
    X_test_tensor = torch.from_numpy(X_test).float()

    # [3] Build & load the model
    model = Net(
        in_dim=in_dim,
        hidden_dims=hidden_dims,
        out_dim=out_dim,
        activation_str=activation
    )
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()

    # [4] Inference
    with torch.no_grad():
        Y_pred_scaled = model(X_test_tensor).numpy()

    # [5] Inverse-transform to original scale
    Y_pred = y_scaler.inverse_transform(Y_pred_scaled)
    Y_true = y_scaler.inverse_transform(Y_test_scaled)

    # [6] Compute R² for the first output
    r2_first = r2_score(Y_true[:, 0], Y_pred[:, 0])
    print(f"R² score for first output ({output_cols[0]}): {r2_first:.4f}")

    # [7] Plot histogram of errors for each output
    plt.figure(figsize=(15, 10))
    for i, col_name in enumerate(output_cols):
        err = Y_pred[:, i] - Y_true[:, i]
        plt.subplot(2, 3, i + 1)
        plt.hist(err, bins=30, edgecolor='k', alpha=0.7)
        plt.title(f'Output {i+1} ({col_name}): Pred − Actual')
        plt.xlabel('Error')
        plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # [8] Plot Actual vs. Predicted scatter for each output
    plt.figure(figsize=(15, 10))
    for i, col_name in enumerate(output_cols):
        actual = Y_true[:, i]
        pred   = Y_pred[:, i]
        mn, mx = actual.min(), actual.max()
        mn, mx = min(mn, pred.min()), max(mx, pred.max())

        plt.subplot(2, 3, i + 1)
        plt.scatter(actual, pred, s=10, alpha=0.6)
        plt.plot([mn, mx], [mn, mx], 'r--', linewidth=1)
        plt.title(f'Output {i+1} ({col_name}): Actual vs Pred')
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.xlim(mn, mx)
        plt.ylim(mn, mx)

    plt.tight_layout()
    plt.show()